In [2]:
import os
import torch
from torch.utils.data import dataloader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")



using device: cpu


In [3]:
#defining the class
class NeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = torch.nn.Flatten()
        self.linear_relu_stack = torch.nn.Sequential(
            torch.nn.Linear(28*28, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 512),
            torch.nn.ReLU(),
            torch.nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
model = NeuralNetwork().to(device)
print(model)


NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [21]:
#training the model
#loss function
loss_fn =torch.nn.CrossEntropyLoss()

#optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [15]:
#split data into batches
batch_size = 64
train_dataloader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        root="data",
        train=True,
        download=True,
        transform=transforms.Compose([transforms.ToTensor()]),
    ),
    batch_size=batch_size,
    shuffle=True,
    )

val_dataloader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        root="data",
        train=False,
        download=True,
        transform=transforms.Compose([transforms.ToTensor()]),
    ),
    batch_size=batch_size,
    shuffle=True,
    )

print(f"length of the training dataloader: {len(train_dataloader)}")
print(f"length of the validation dataloader: {len(val_dataloader)}")

test_dataloader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        root="data",
        train=False,
        download=True,
        transform=transforms.Compose([transforms.ToTensor()])
    ),
    batch_size=batch_size,
    shuffle=True,
    )

print(f"length of the test dataloader: {len(test_dataloader)}")

# Helper function to move batch to device
def batch_to_device(X, y, device):
    return X.to(device), y.to(device)

# Training loop for one epoch
def train_loader(train_dataloader, model, loss_fn, optimizer, device):
    model.train()
    for X, y in train_dataloader:
        X, y = batch_to_device(X, y, device)
        pred = model(X)
        loss = loss_fn(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

length of the training dataloader: 938
length of the validation dataloader: 157
length of the test dataloader: 157


In [22]:
#training loop
def train_model(train_dataloader, val_dataloader, model, loss_fn, optimizer, device):
    size = len(train_dataloader)
    for X, y in train_dataloader:
        X, y = batch_to_device(X, y, device)
        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


#evaluation loop
def eval(dataloader, model, loss_fn, device):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = batch_to_device(X, y, device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


def on_validation_data(dataloader, model, loss_fn, device):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    val_loss = 0
    correct = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = batch_to_device(X, y, device)
            pred = model(X)
            val_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    val_loss /= num_batches
    correct /= size
    print(f"Validation Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {val_loss:>8f} \n")



#number of epochs
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_model(train_dataloader, val_dataloader, model, loss_fn, optimizer, device)
    eval(test_dataloader, model, loss_fn, device)
    on_validation_data(val_dataloader, model, loss_fn, device)

print("Done!")
    

Epoch 1
-------------------------------
Test Error: 
 Accuracy: 90.3%, Avg loss: 0.540471 

Validation Error: 
 Accuracy: 90.3%, Avg loss: 0.543618 

Epoch 2
-------------------------------
Test Error: 
 Accuracy: 90.6%, Avg loss: 0.562024 

Validation Error: 
 Accuracy: 90.6%, Avg loss: 0.565020 

Epoch 3
-------------------------------
Test Error: 
 Accuracy: 90.3%, Avg loss: 0.578639 

Validation Error: 
 Accuracy: 90.3%, Avg loss: 0.582087 

Epoch 4
-------------------------------
Test Error: 
 Accuracy: 90.5%, Avg loss: 0.591958 

Validation Error: 
 Accuracy: 90.5%, Avg loss: 0.598365 

Epoch 5
-------------------------------
Test Error: 
 Accuracy: 90.3%, Avg loss: 0.612014 

Validation Error: 
 Accuracy: 90.3%, Avg loss: 0.610114 

Epoch 6
-------------------------------
Test Error: 
 Accuracy: 90.5%, Avg loss: 0.616595 

Validation Error: 
 Accuracy: 90.5%, Avg loss: 0.614810 

Epoch 7
-------------------------------
Test Error: 
 Accuracy: 90.5%, Avg loss: 0.640177 

Validati